In [0]:
# Databricks notebook source
# PATH: nyc-taxi-pipeline/notebooks/01_bronze_ingest.py

import io
import urllib.request
import pandas as pd
from pyspark.sql.functions import current_timestamp, lit

# 1. Define input parameters/widgets for Airflow integration
dbutils.widgets.text("load_type", "incremental") 
dbutils.widgets.text("year", "2026")
dbutils.widgets.text("month", "01")

# 2. Extract and format parameters
load_type = dbutils.widgets.get("load_type")
year = dbutils.widgets.get("year")
month = f"{int(dbutils.widgets.get('month')):02d}"

# 3. Create flat traditional database namespace
print("Initializing database assets...")
spark.sql("CREATE DATABASE IF NOT EXISTS nyc_taxi_medallion")

# 4. Construct remote source URL based on pipeline logic
if load_type == "base":
    print("Executing Initial Base Load...")
    source_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-12.parquet"
else:
    print(f"Executing Incremental Load for Execution Period: {year}-{month}")
    source_url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{year}-{month}.parquet"

# 5. In-Memory Streaming Bypass
print(f"Downloading remote file into memory from: {source_url}")

req = urllib.request.Request(
    source_url, 
    headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
)

with urllib.request.urlopen(req) as response:
    # Read the raw byte array stream directly into an in-memory bytes buffer
    file_bytes = response.read()

print("Download completed. Parsing Parquet data structure in memory...")

# 6. Read the bytes into a Pandas DataFrame using the pyarrow engine
# This completely circumvents 'file:/' protocols and filesystem permission checks
pdf_raw = pd.read_parquet(io.BytesIO(file_bytes), engine='pyarrow')

print(f"Successfully loaded {len(pdf_raw)} rows into driver memory.")

# 7. Convert the Pandas DataFrame directly to a native Spark DataFrame
# Databricks optimizes this conversion internally via Apache Arrow
df_raw = spark.createDataFrame(pdf_raw)

# 8. Transform data with operational metadata auditing columns
df_bronze = df_raw.withColumn("ingest_timestamp", current_timestamp()) \
                  .withColumn("source_year", lit(year)) \
                  .withColumn("source_month", lit(month))

# 9. Append clean Delta record batch into the managed Bronze target table
target_table = "nyc_taxi_medallion.bronze_yellow_trips"
print(f"Appending ingestion batch to delta table: {target_table}")

df_bronze.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable(target_table)

print("Bronze data successfully ingested!")